In [1]:

import pandas as pd

dataset = pd.read_csv("open_data/bzkopen_addresses_val.csv")



In [5]:
import transformers
import time
from transformers import pipeline

class LlamaAddressSegmentationModel:
    def __init__(self, model_name, prompt, output_parser = None, batch_size=8):
        tokenizer = transformers.AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct", padding_side='left')
        self.pipe = pipeline("text-generation", model=model_name, batch_size=batch_size, tokenizer=tokenizer)
        self.pipe.tokenizer.pad_token_id = self.pipe.model.config.eos_token_id[0]
        self.prompt = prompt
        self.output_parser = output_parser or (lambda x : x)

    def segment_addresses(self, addresses : list[str]) -> str:
        messages = [[
            {"role": "user", "content": self.prompt.format(address=address)}
        ] for address in addresses]
        result = self.pipe(messages)
        responses = [self.output_parser(r[0]["generated_text"][1]["content"]) for r in result]
        return responses

addresses = dataset["FullAddress"].sample(frac=1, random_state=42).tolist()

def test_batch_sizes(batch_sizes):
    results = []
    for batch_size in batch_sizes:
        if batch_size > len(addresses):
            print("Skipping batch size", batch_size, "as it exceeds dataset size")
            continue
        model = LlamaAddressSegmentationModel(
            model_name="meta-llama/Llama-3.2-1B-Instruct",
            prompt="Segment the following address into its components: {address}",
            batch_size=batch_size
        )
        start = time.monotonic()
        responses = model.segment_addresses(addresses[:batch_size])
        deltatime = (time.monotonic() - start)
        print(f"Batch size: {batch_size}, Time taken: {deltatime} seconds")
        results.append((batch_size, deltatime, deltatime / batch_size))
    print(f"Best batch size: {min(results, key=lambda x: x[2])}")
test_batch_sizes([1, 2, 4, 8, 16, 32, 64, 128])

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Batch size: 32, Time taken: 34.3130000000001 seconds


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Batch size: 64, Time taken: 39.5619999999999 seconds


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Batch size: 128, Time taken: 165.90600000000086 seconds
Best batch size: (64, 39.5619999999999, 0.6181562499999984)
